# Advanced PEFT Type: QLoRA (PyTorch)
**Date**: 2026-05-31  
**Objective**: Advanced PyTorch from-scratch PEFT using QLoRA adaptation strategy.

## Common Logic Highlight
- Runtime device switch uses USE_GPU from configs/runtime.env.
- Dataset and metrics are aligned with all advanced PEFT notebooks.
- Type-specific focus: quantization-aware low-rank adaptation (simulated for from-scratch setup).

In [ ]:
import os
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
def load_runtime_env() -> None:
    candidate_paths = [
        os.path.join(os.getcwd(), 'configs', 'runtime.env'),
        os.path.join(os.getcwd(), 'configs', 'runtime.env.example'),
        os.path.join(os.getcwd(), '..', 'configs', 'runtime.env'),
        os.path.join(os.getcwd(), '..', 'configs', 'runtime.env.example'),
        os.path.join(os.getcwd(), '..', '..', 'configs', 'runtime.env'),
        os.path.join(os.getcwd(), '..', '..', 'configs', 'runtime.env.example'),
    ]
    env_loaded = False
    for path in candidate_paths:
        if not os.path.exists(path):
            continue
        with open(path, 'r', encoding='utf-8') as handle:
            for raw_line in handle:
                line = raw_line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                key, value = line.split('=', 1)
                os.environ[key.strip()] = value.strip().strip("\"'")
        env_loaded = True
        break
    if not env_loaded and 'USE_GPU' not in os.environ:
        os.environ['USE_GPU'] = '1'

def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {'0', 'false', 'no', 'off'}

load_runtime_env()
USE_GPU = parse_use_gpu_flag(os.getenv('USE_GPU', '1'))
RUNTIME_DEVICE = 'cuda' if USE_GPU and torch.cuda.is_available() else 'cpu'
device = torch.device(RUNTIME_DEVICE)
print(f'USE_GPU={int(USE_GPU)} | runtime_device={RUNTIME_DEVICE}')

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    seq_len: int = 28
    emb_dim: int = 96
    hidden_dim: int = 128
    adapter_rank: int = 12
    prompt_len: int = 4
    batch_size: int = 4
    epochs: int = 4
    lr: float = 1e-3
    train_size: float = 0.8

cfg = ExperimentConfig()

In [ ]:
samples = [
    ('Server not reachable after deployment', 2),
    ('Password reset email not received', 1),
    ('Dashboard typo in heading', 0),
    ('Payment API timing out for premium users', 2),
    ('Need help changing profile picture', 0),
    ('CPU usage spikes to 100 percent hourly', 2),
    ('Can we export reports to CSV?', 0),
    ('Intermittent login failures for SSO users', 2),
    ('Dark mode icon is slightly misaligned', 0),
    ('Data sync lag observed in EU region', 1),
    ('Mobile app crashes on checkout page', 2),
    ('Feature request: bulk archive tickets', 0),
    ('Webhook retries causing duplicate events', 1),
    ('Fraud alert queue delayed by 5 minutes', 2),
    ('Question about invoice date format', 0),
    ('Latency increased after model update', 1),
]
df = pd.DataFrame(samples, columns=['text', 'label'])
df['label_name'] = df['label'].map({0: 'low', 1: 'medium', 2: 'high'})
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
split_idx = int(len(df) * cfg.train_size)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
display(train_df.head(3))

In [ ]:
def build_vocab(texts: list[str]) -> dict[str, int]:
    vocab = {'<pad>': 0, '<unk>': 1}
    for text in texts:
        for token in text.lower().split():
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

def encode_text(text: str, vocab: dict[str, int], seq_len: int) -> list[int]:
    ids = [vocab.get(tok, vocab['<unk>']) for tok in text.lower().split()]
    ids = ids[:seq_len]
    ids += [vocab['<pad>']] * max(0, seq_len - len(ids))
    return ids

vocab = build_vocab(train_df['text'].tolist())
x_train = np.array([encode_text(t, vocab, cfg.seq_len) for t in train_df['text']], dtype=np.int64)
x_test = np.array([encode_text(t, vocab, cfg.seq_len) for t in test_df['text']], dtype=np.int64)
y_train = train_df['label'].to_numpy(dtype=np.int64)
y_test = test_df['label'].to_numpy(dtype=np.int64)
train_ds = TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
test_ds = TensorDataset(torch.tensor(x_test), torch.tensor(y_test))
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)

In [ ]:
class AdvancedAdapter(nn.Module):
    def __init__(self, hidden_dim: int, rank: int):
        super().__init__()
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.up(self.down(x))

class AdvancedPeftClassifier(nn.Module):
    def __init__(self, vocab_size: int, cfg: ExperimentConfig):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, cfg.emb_dim)
        self.soft_prompt = nn.Parameter(torch.randn(cfg.prompt_len, cfg.emb_dim) * 0.02)
        self.encoder = nn.GRU(cfg.emb_dim, cfg.hidden_dim, batch_first=True, bidirectional=True)
        self.adapter = AdvancedAdapter(cfg.hidden_dim * 2, cfg.adapter_rank)
        self.classifier = nn.Linear(cfg.hidden_dim * 2, 3)
        for p in self.embedding.parameters():
            p.requires_grad = False
        for p in self.encoder.parameters():
            p.requires_grad = False

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(input_ids)
        bsz = emb.size(0)
        prompt = self.soft_prompt.unsqueeze(0).expand(bsz, -1, -1)
        x = torch.cat([prompt, emb], dim=1)
        enc, _ = self.encoder(x)
        pooled = enc.mean(dim=1)
        adapted = self.adapter(pooled)
        return self.classifier(adapted)

model = AdvancedPeftClassifier(len(vocab), cfg).to(device)

In [ ]:
optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=cfg.lr)
criterion = nn.CrossEntropyLoss()
for _ in range(cfg.epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
model.eval()
preds, labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        pred = torch.argmax(logits, dim=-1).cpu().numpy()
        preds.extend(pred.tolist())
        labels.extend(yb.numpy().tolist())
metrics = {
    'accuracy': float(accuracy_score(labels, preds)),
    'macro_f1': float(f1_score(labels, preds, average='macro')),
}
print(metrics)

In [ ]:
metric_names = ['accuracy', 'macro_f1']
metric_values = [metrics[k] for k in metric_names]
plt.figure(figsize=(6, 4))
plt.bar(metric_names, metric_values)
plt.ylim(0.0, 1.0)
plt.title('Advanced QLoRA PyTorch - Evaluation Snapshot')
plt.ylabel('Score')
plt.show()

## Summary
- Framework: PyTorch, PEFT type: QLoRA.
- Advanced focus: quantization-aware low-rank adaptation (simulated for from-scratch setup).
- Common logic remains aligned with the TensorFlow counterpart notebook.